# progettoML — TabPFN-2.5 + LoRA (dominio finanziario)

Esegui le celle **in ordine**. Dopo la cella di setup, riavvia la sessione come indicato.

## 1. Setup (una volta sola)

In [ ]:
!git clone https://github.com/VittorioIusi/progettoML.git
%cd progettoML
!pip install -q -e ./TabPFN-main
!pip install -q -r requirements.txt
print('FATTO. Ora: Runtime -> Riavvia sessione, poi parti dalla cella 2.')

## 2. Token + verifica ambiente (dopo il riavvio)

In [ ]:
import os
from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')

import tabpfn, torch
print('tabpfn:', tabpfn.__version__, '| atteso 8.0.8')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 3. Aggiorna il repo e importa

`git pull` porta nel runtime l'ultima versione del codice (con i dataset finanziari e `run_lora_datasize_financial`).

In [ ]:
%cd /content/progettoML
!git pull

import sys; sys.path.insert(0, '/content/progettoML')
from models import run_lora_exp5, run_lora_ablation, run_lora_datasize_financial
from models.lora import LoRAConfig
print('Import OK')

## Esperimento 1 — Generalizzazione in-domain (finanziario)

Un LoRA allenato sui 5 dataset finanziari, valutato sui 4 di test in-domain.

In [ ]:
df1, _ = run_lora_exp5(device='cuda', epochs=50, save_path='lora_financial.pt')

## Esperimento 2 — Ablation sulla capacità

Tre configurazioni crescenti: se restano tutte neutre, TabPFN è al soffitto.

In [ ]:
configs = [
    LoRAConfig(r=8,  alpha=16, target_modules=('q_projection','v_projection')),
    LoRAConfig(r=16, alpha=32, target_modules=('q_projection','v_projection')),
    LoRAConfig(r=16, alpha=32, target_modules=('q_projection','k_projection','v_projection','out_projection')),
]
df2 = run_lora_ablation(configs, ['r8_qv','r16_qv','r16_qkvo'], epochs=50, device='cuda')

## Esperimento 3 — Ablation sulla dimensione dei dati (gmsc)

Test set **fisso** di 8.000 righe; training crescenti 500 -> 40.000.
Produce tabella + grafico `results/datasize_financial_auc.png`.
Lungo: ~2-3h su T4 (7 dimensioni x base+LoRA).

In [ ]:
df3 = run_lora_datasize_financial(device='cuda')

## (Opzionale) Pipeline esterna — Esp. 4 e 5

Ensembling, calibrazione e soglia. Lasciano i pesi congelati: indipendenti dal LoRA.

In [ ]:
from pipeline.external_pipeline import (
    run_calibration_threshold_experiment, run_ensemble_experiment)

agg4 = run_calibration_threshold_experiment(
    device='cuda', n_seeds=5, save_path='results/exp4_calib_threshold.csv')
agg3 = run_ensemble_experiment(
    device='cuda', n_seeds=3, n_ensemble=5, save_path='results/exp3_ensemble.csv')